In [1]:
!pip -q install --upgrade pip
!pip -q install torch transformers sentence-transformers faiss-cpu accelerate

# %%
import os
import json
import math
import time
from typing import List, Tuple

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import faiss

DATA_PATH = "data/nutuk.pdf"   # <- Nutuk tam metin dosyan
ARTIFACT_DIR = "artifacts"
INDEX_PATH = os.path.join(ARTIFACT_DIR, "faiss.index")
CHUNKS_PATH = os.path.join(ARTIFACT_DIR, "chunks.jsonl")

EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
GEN_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# RAG parametreleri
CHUNK_SIZE_WORDS = 220
CHUNK_OVERLAP_WORDS = 40
TOP_K = 5
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.2
TOP_P = 0.9

os.makedirs(ARTIFACT_DIR, exist_ok=True)

# Cihaz tespiti
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Cihaz: {DEVICE}")



[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
C:\Users\ilydt\AppData\Local\Programs\Python\Python312\python.exe -m pip -q install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Cihaz: cuda


In [2]:
def chunk_text_by_words(text: str, chunk_size: int, overlap: int) -> List[str]:
    """Basit, sağlam parçalayıcı: kelime bazlı sabit boy parçalar (overlap ile)."""
    # Paragrafları normalize et
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    # Aşırı boşluk temizliği
    text = "\n".join([p.strip() for p in text.split("\n")])

    words = text.split()
    chunks = []
    i = 0
    n = len(words)
    while i < n:
        end = min(i + chunk_size, n)
        chunk_words = words[i:end]
        chunk = " ".join(chunk_words).strip()
        if chunk:
            chunks.append(chunk)
        if end == n:
            break
        i = end - overlap  # overlap geri sarma
        if i < 0:
            i = 0
    return chunks


In [3]:
def build_or_load_index(text_path: str,
                        embed_model_name: str,
                        index_path: str,
                        chunks_path: str,
                        chunk_size: int = 220,
                        overlap: int = 40) -> Tuple[faiss.IndexFlatIP, List[str], SentenceTransformer]:
    """
    Nutuk metnini yükleyip chunk'lara böler, embedding'leri hesaplar ve FAISS IP index oluşturur.
    Diskte varsa tekrar kullanır.
    """
    embedder = SentenceTransformer(embed_model_name, device=DEVICE)
    embedder.max_seq_length = 512

    if os.path.exists(index_path) and os.path.exists(chunks_path):
        print("[FAISS] Mevcut indeks ve chunk'lar yükleniyor…")
        # FAISS
        faiss_index = faiss.read_index(index_path)
        # Chunks
        chunks = []
        with open(chunks_path, "r", encoding="utf-8") as f:
            for line in f:
                chunks.append(json.loads(line)["text"])
        return faiss_index, chunks, embedder

    if not os.path.exists(text_path):
        raise FileNotFoundError(
            f"Nutuk metni bulunamadı: {text_path}\n'**data/nutuk.txt**' yoluna yerleştirdiğinden emin ol.")

    print("[DATA] Nutuk metni okunuyor ve parçalanıyor…")
    with open(text_path, "r", encoding="utf-8") as f:
        full_text = f.read()

    chunks = chunk_text_by_words(full_text, chunk_size, overlap)
    print(f"[DATA] Toplam {len(chunks)} parça üretildi.")

    print("[EMB] Embedding hesaplanıyor… (biraz sürebilir)")
    embeddings = embedder.encode(chunks, batch_size=64, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)

    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    faiss.write_index(index, index_path)
    with open(chunks_path, "w", encoding="utf-8") as f:
        for ch in chunks:
            f.write(json.dumps({"text": ch}, ensure_ascii=False) + "\n")

    print("[FAISS] İndeks ve chunk'lar diske kaydedildi.")
    return index, chunks, embedder


In [4]:
def retrieve(query: str, index: faiss.IndexFlatIP, embedder: SentenceTransformer, chunks: List[str], top_k: int = 5) -> List[Tuple[int, float, str]]:
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, ids = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue
        results.append((int(idx), float(score), chunks[int(idx)]))
    return results

In [5]:
def load_llm(model_name: str = GEN_MODEL_NAME):
    print(f"[LLM] {model_name} modeli yükleniyor…")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        device_map="auto" if DEVICE == "cuda" else None
    )
    if DEVICE == "cpu":
        model = model.to(DEVICE)
    return tokenizer, model

In [6]:
SYSTEM_PROMPT = (
    "Sen Nutuk üzerine soru cevaplayan yardımcı bir asistansın. "
    "Yalnızca verilen bağlama dayanarak yanıt ver. Bağlamda yoksa 'Bilmiyorum' de. "
    "Gerektiğinde metinden kısa alıntılar yapabilirsin. Türkçe, net ve öğretici bir dille yaz."
)


def build_context_block(retrieved: List[Tuple[int, float, str]]) -> str:
    blocks = []
    for i, (idx, score, chunk) in enumerate(retrieved, start=1):
        blocks.append(f"[Parça {i} | skor={score:.3f}]\n{chunk}")
    return "\n\n".join(blocks)


def generate_answer(question: str,
                    tokenizer: AutoTokenizer,
                    model: AutoModelForCausalLM,
                    context_text: str,
                    temperature: float = TEMPERATURE,
                    top_p: float = TOP_P,
                    max_new_tokens: int = MAX_NEW_TOKENS) -> str:

    user_prompt = (
        f"\n\n[BAĞLAM]\n{context_text}\n\n"
        f"[GÖREV]\nYukarıdaki bağlamı kullanarak şu soruyu yanıtla: {question}\n\n"
        f"Kurallar: (1) Bağlam dışına çıkma, (2) Kısa ve açık yaz, (3) Emin değilsen 'Bilmiyorum' de."
    )

    # Chat template (varsa) kullan
    if hasattr(tokenizer, "apply_chat_template"):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]
        model_input = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt")
    else:
        # Basit fallback
        full_prompt = f"<s>[SYSTEM]\n{SYSTEM_PROMPT}\n[/SYSTEM]\n[USER]\n{user_prompt}\n[/USER]\nAssistant:"
        model_input = tokenizer(full_prompt, return_tensors="pt")

    model_input = {k: v.to(model.device) for k, v in model_input.items()}

    with torch.no_grad():
        out = model.generate(
            **model_input,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id
        )

    text = tokenizer.decode(out[0], skip_special_tokens=True)

    # Eğer chat template kullanıldıysa, sadece asistan cevabını ayıklayalım
    if hasattr(tokenizer, "apply_chat_template"):
        # Qwen formatında son kısımdaki assistant mesajını bulmaya çalış
        # Basit bir ayıklama: model_output = text.split("assistant\n")[-1]
        return text.split("assistant\n")[-1].strip() if "assistant\n" in text else text.strip()
    return text.strip()

In [7]:
# PDF dosyasını metne çevirip RAG chatbot ile kullanma
import os
import time

PDF_PATH = "data/nutuk.pdf"  # Nutuk PDF dosyası
TXT_PATH = "data/nutuk.txt"  # PDF'ten çıkarılacak metin dosyası

# Klasör var mı kontrol et, yoksa oluştur
os.makedirs(os.path.dirname(TXT_PATH), exist_ok=True)

# Gerekli paket yükleme
!pip install --quiet PyMuPDF transformers torch sentence-transformers

import fitz  # PyMuPDF
import torch

# PDF'ten metin çıkarma fonksiyonu
def pdf_to_text(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

# PDF'i metin dosyasına kaydet
with open(TXT_PATH, "w", encoding="utf-8") as f:
    f.write(pdf_to_text(PDF_PATH))

print(f"Nutuk PDF'den txt'ye dönüştürüldü: {TXT_PATH}")

DATA_PATH = TXT_PATH  # RAG kodu bundan sonra txt dosyasını kullanacak

# =============================
# Düzeltilmiş generate_answer fonksiyonu (AttributeError fix)
def generate_answer_fixed(question, tokenizer, model, context_text, temperature=0.2, top_p=0.9, max_new_tokens=512):
    prompt = f"{context_text}\nSoru: {question}\nCevap:"
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs['input_ids'].to(model.device)
    attention_mask = inputs['attention_mask'].to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# =============================
# Düzeltilmiş answer_question_rag fonksiyonu
def answer_question_rag(question: str,
                        top_k: int = TOP_K,
                        temperature: float = TEMPERATURE,
                        top_p: float = TOP_P,
                        max_new_tokens: int = MAX_NEW_TOKENS) -> dict:
    start = time.time()
    index, chunks, embedder = build_or_load_index(
        DATA_PATH,
        EMBEDDING_MODEL_NAME,
        INDEX_PATH,
        CHUNKS_PATH,
        CHUNK_SIZE_WORDS,
        CHUNK_OVERLAP_WORDS,
    )
    retrieved = retrieve(question, index, embedder, chunks, top_k=top_k)
    context_block = build_context_block(retrieved)

    tokenizer, model = load_llm(GEN_MODEL_NAME)
    answer = generate_answer_fixed(
        question,
        tokenizer,
        model,
        context_block,
        temperature=temperature,
        top_p=top_p,
        max_new_tokens=max_new_tokens,
    )
    elapsed = time.time() - start

    return {
        "question": question,
        "answer": answer,
        "top_k": top_k,
        "retrieved_count": len(retrieved),
        "elapsed_sec": round(elapsed, 2)
    }



[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Nutuk PDF'den txt'ye dönüştürüldü: data/nutuk.txt


In [ ]:
# Örnek kullanım
try:
    demo = answer_question_rag(
        "Atatürk Samsun'a ne zaman çıktı?",
        top_k=5,
        temperature=0.2,
        top_p=0.9,
        max_new_tokens=200,
    )
    print("Soru:", demo["question"]) 
    print("\nCevap:\n", demo["answer"]) 
    #print(f"\nBilgi: top_k={demo['top_k']} | getirilen={demo['retrieved_count']} | süre={demo['elapsed_sec']} sn")
except FileNotFoundError as e:
    print(str(e))


[FAISS] Mevcut indeks ve chunk'lar yükleniyor…
[LLM] Qwen/Qwen2.5-0.5B-Instruct modeli yükleniyor…
